In [ ]:
import numpy as np
import pandas as pd
import os
import re
from tqdm import tqdm
from collections import Counter

import transformers as tf
import spacy

!python -m spacy download en_core_web_lg
nlp = spacy.load("en_core_web_lg")

import warnings
warnings.filterwarnings("ignore")

# For local env
path_prefix = '../data/processed/'


In [ ]:

# For CoLab env


## Loading data

In [ ]:
full_data = pd.read_csv(f'{path_prefix}stories.csv')

full_data.head(2)


In [ ]:
# demo_text = full_data['story'].values[42]

demo_text = full_data.sort_values('word_count')['story'][0]
demo_text


## Loading functions

### Preprocessing

In [ ]:
def line_to_sents(line: str):
    """
    Segment each line to sentences (with `spacy`)
    - Input: `str` (story line)
    - Output: `list` (sents of story line)
    """
    line = nlp(line)
    sents = list(line.sents)
    for i, s in enumerate(sents):
        sents[i] = s.text
    return sents


def line_to_ngrams(line: str, n: int):
    """
    Segment each line to ngrams (with `spacy`)
    - Input: `str` (story line)
    - Output: `list` (ngrams of story line)
    """
    line = nlp(line)
    tokens = [token.text for token in line]
    if tokens.__len__() < n:
        ngrams = [tokens]
    else:
        ngrams = [tokens[i : i + n] for i in range(len(tokens) - n + 1)]
    return ngrams


### Readability

In [ ]:
!pip install textstat
!pip install wordfreq

from textstat import flesch_kincaid_grade as fkg
from textstat import smog_index as smog
from textstat import automated_readability_index as ari
from wordfreq import word_frequency as wf
from wordfreq import zipf_frequency as zf


In [ ]:
# Length-based metrics
def read_scores(text: str):
    """
    Return three readability scores:
    - (Inverted) Flesch-Kincaid grade
    - SMOG score
    - ARI score
    """

    def afkg(text: str):
        return -fkg(text) + 100

    return (afkg(text), smog(text), ari(text))


# Complexity-based metrics: lexical
def lexicon_freq(text: str):
    """
    Return a frequency lexicon from a text
    - Input: `str` (text)
    - Output: `Counter`
    """
    doc = nlp(text)
    tokens = [
        token.text for token in doc if token.is_punct != True and token.has_vector
    ]
    lexicon_freq = Counter(tokens)
    return lexicon_freq


def lfp_text(text: str, threshold=3.0, verbose=False):
    """
    Return the LFP (proportion of low-freq words) from a text. Low-freq words are those words whose Zipf frequency is below 3.0 by default.
    - Input: `str` (text)
    - Output: `float`
    """
    doc = nlp(text)
    tokens = [
        token.text for token in doc if token.is_punct != True and token.has_vector
    ]
    token_count = tokens.__len__()

    def is_lfword(word: str, threshold=threshold):
        return zf(word, "en", wordlist="large") <= threshold

    lf_words = [t for t in tokens if is_lfword(t)]
    lf_count = lf_words.__len__()

    lfp = lf_count / token_count
    if verbose:
        print(f"Token count: {token_count}")
        print(f"LF word count: {lf_count}")
        print(f"LF word ratio: {lfp:.2f}")
    return lfp


# Complexity-based metrics: syntactic
def mdd_sent(sent: str):
    """
    Return the MDD of a sentence.
    - Input: `str` (sent)
    - Output: `float`
    """

    def dep_dict(sent: str):
        doc = nlp(sent)
        dep_dict = {"sent": sent, "token_ind": [], "head_ind": [], "dep_d": []}
        n_punct_before = [int(token.pos_ == "PUNCT") for token in doc]
        n_space_before = [int(token.pos_ == "SPACE") for token in doc]

        for token in doc:
            if token.pos_ != "PUNCT":
                if token.pos_ != "SPACE":
                    dep_dict["token_ind"].append(
                        token.i
                        - sum(n_punct_before[: token.i])
                        - sum(n_space_before[: token.i])
                    )
                    dep_dict["head_ind"].append(
                        token.head.i
                        - sum(n_punct_before[: token.head.i])
                        - sum(n_space_before[: token.head.i])
                    )

        dep_dict["dep_d"] = list(
            np.array(dep_dict["token_ind"]) - np.array(dep_dict["head_ind"])
        )
        return dep_dict

    d_dict = dep_dict(sent)
    mdd = np.mean(np.array([abs(i) for i in d_dict["dep_d"] if i]))
    return mdd


def mdd_text(text: str):
    """
    Return the MDD of a text.
    - Input: `str` (text)
    - Output: `np.array`
    """
    sents = line_to_sents(text)
    mdds = np.array([])
    for sent in sents:
        m = mdd_sent(str(sent))
        if not np.isnan(m):
            mdds = np.append(mdds, m)
        else:
            mdds = np.append(mdds, 0)
    return mdds

print(read_scores(demo_text))
print(lfp_text(demo_text))
print(mdd_text(demo_text))


In [ ]:
from transformers import GPTNeoXForCausalLM, AutoTokenizer
import torch

# pythia_size = '6.9b'
pythia_size = '6.9b'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load model with optimized settings for GPU
model = GPTNeoXForCausalLM.from_pretrained(
  f"EleutherAI/pythia-{pythia_size}-deduped",
  revision="step3000",
  cache_dir=f"./pythia-{pythia_size}-deduped/step3000",
  torch_dtype=torch.float16,  # Use half precision to save memory
  device_map="auto",  # Automatically split model across available GPUs
).to(device)

# Enable evaluation mode and other optimizations
model.eval()
torch.cuda.empty_cache()  # Clear GPU cache

tokenizer = AutoTokenizer.from_pretrained(
  f"EleutherAI/pythia-{pythia_size}-deduped",
  revision="step3000",
  cache_dir=f"./pythia-{pythia_size}-deduped/step3000",
)


In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
torch.cuda.set_device(device)
model = model.to(device)

def prepare_inputs(text):
    """预处理文本并返回设备上的输入张量"""
    for token in nlp(text):
        if token.has_vector == False:
            text = text.replace(token.text, "[UNK]")
        if token.is_punct:
            text = text.replace(token.text, "")

    inputs = tokenizer(text, return_tensors="pt")
    return {key: value.to(device) for key, value in inputs.items()}

def token_surprisal(text: str, verbose=False):
    inputs = prepare_inputs(text)
    input_ids = inputs["input_ids"]
    outputs = model(**inputs, labels=input_ids)
    logits = outputs.logits
    logprobs = torch.gather(
        torch.nn.functional.log_softmax(logits, dim=2), 2, input_ids.unsqueeze(2)
    )
    return -logprobs[0, -1, 0].item()

def context_prob(target_word, context_words, verbose=False):
    text = " ".join(context_words) + " " + target_word
    inputs = prepare_inputs(text)
    num_tokens = inputs["input_ids"].shape[1]

    if num_tokens <= 1:
        return token_surprisal(text)

    input_ids = inputs["input_ids"][:, :-1]
    output_ids = inputs["input_ids"][:, -1].unsqueeze(0)
    outputs = model(input_ids, labels=input_ids)
    logits = outputs.logits
    probs = torch.gather(
        torch.nn.functional.softmax(logits, dim=2), 2, output_ids.unsqueeze(2)
    )
    return probs[0, -1, 0].item()

# 测试
print(context_prob('doctor', ['I', 'am', 'a']))
print(context_prob('egg', ['I', 'am', 'an']))


In [ ]:
def prepare_inputs(text):
    """预处理文本并返回设备上的输入张量"""
    for token in nlp(text):
        if token.has_vector == False:
            text = text.replace(token.text, "[UNK]")
        if token.is_punct:
            text = text.replace(token.text, "")

    inputs = tokenizer(text, return_tensors="pt")
    return {key: value.to(device) for key, value in inputs.items()}

def sent_surprisal(text: str, verbose=False):
    inputs = prepare_inputs(text)
    num_tokens = inputs["input_ids"].shape[1]

    if num_tokens <= 1:
        return token_surprisal(text)

    input_ids = inputs["input_ids"]
    outputs = model(**inputs, labels=input_ids)
    logits = outputs.logits

    logprobss = []
    for i in range(1, num_tokens):
        output_ids = inputs["input_ids"][:, i].unsqueeze(0)
        logprobs = torch.gather(
            torch.nn.functional.log_softmax(logits, dim=2), 2, output_ids.unsqueeze(2)
        )
        logprobss.append(-logprobs[0, -1, 0].item())
        if verbose:
            print(f"Token {i}: {logprobss[-1]}")

    return logprobss

def ngram_surprisal(text: str, n=3, verbose=False):
    inputs = prepare_inputs(text)
    num_tokens = inputs["input_ids"].shape[1]

    if num_tokens <= n:
        return token_surprisal(text)

    input_ids = inputs["input_ids"]
    outputs = model(**inputs, labels=input_ids)
    logits = outputs.logits

    logprobss = []
    for i in range(n, num_tokens):
        output_ids = inputs["input_ids"][:, i].unsqueeze(0)
        logprobs = torch.gather(
            torch.nn.functional.log_softmax(logits, dim=2), 2, output_ids.unsqueeze(2)
        )
        logprobss.append(-logprobs[0, -1, 0].item())
        if verbose:
            print(f"N-gram {i-n}:{i} -> {logprobss[-1]}")
    return logprobss

def ssrp_text(text: str, verbose=False):
    """
    Return the sentence-based surprisal of the text.
    (Recommmended to use this function on a sentence level)
    - Input: `str` (text)
    - Output: `np.array`
    """
    sents = line_to_sents(text)
    srps = np.array([])
    for s in sents:
        try:
            srp = np.mean(sent_surprisal(s))
        except Exception as e:
            if verbose:
                print(f"Error processing sentence: {s}, error: {e}")
            srp = 0
        srps = np.append(srps, srp)

    return srps

def nsrp_text(text: str, verbose=False, n=3):
    """
    Return the n-gram surprisal of the text.
    (Dont use this, it is slow, and inaccurate.)
    - Input: `str` (text)
    - Output: `np.array`
    """
    sents = line_to_sents(text)
    srps = np.array([])
    for s in tqdm(sents):
        try:
            srp = np.mean(ngram_surprisal(s, n=n))
        except Exception as e:
            if verbose:
                print(f"Error processing sentence: {s}, error: {e}")
            srp = 0
        srps = np.append(srps, srp)

    return srps

def srp_sent(text: str, verbose=False, n=3):
    """
    Return the n-gram and sentence-based surprisal of the text.
    - Input: `str` (text)
    - Output: `tuple` (n_gram_surprisals, sentence_surprisals)
    """
    inputs = prepare_inputs(text)
    num_tokens = inputs["input_ids"].shape[1]

    with torch.no_grad():  # 减少内存使用
        outputs = model(**inputs, labels=inputs["input_ids"])
        logits = outputs.logits

    if num_tokens <= n:
        logprobs = torch.gather(
            torch.nn.functional.log_softmax(logits, dim=2), 2, inputs["input_ids"].unsqueeze(2)
        )
        n_logprobss = [-logprobs[0, -1, 0].item()]
        if num_tokens <= 1:
            s_logprobss = n_logprobss
            return n_logprobss, s_logprobss

    n_logprobss = []
    for i in range(n, num_tokens):
        output_ids = inputs["input_ids"][:, i].unsqueeze(0)
        logprobs = torch.gather(
            torch.nn.functional.log_softmax(logits, dim=2), 2, output_ids.unsqueeze(2)
        )
        n_logprobss.append(-logprobs[0, -1, 0].item())

    s_logprobss = []
    for i in range(1, num_tokens):
        output_ids = inputs["input_ids"][:, i].unsqueeze(0)
        logprobs = torch.gather(
            torch.nn.functional.log_softmax(logits, dim=2), 2, output_ids.unsqueeze(2)
        )
        s_logprobss.append(-logprobs[0, -1, 0].item())

    return n_logprobss, s_logprobss

def srp_text(text: str, verbose=False, n=3):
    """
    Return the n-gram and sentence-based surprisal of the text.
    - Input: `str` (text)
    - Output: `np.ndarray` with shape (2, num_sentences)
    """
    sents = line_to_sents(text)
    srps = np.ndarray((2, len(sents)))

    for i, s in enumerate(sents):
        try:
            nsrp, ssrp = srp_sent(s, n=n)
            nsrp = np.array(nsrp)
            ssrp = np.array(ssrp)
            srps[0, i] = np.mean(nsrp[~np.isnan(nsrp)]) if len(nsrp) > 0 else np.nan
            srps[1, i] = np.mean(ssrp[~np.isnan(ssrp)]) if len(ssrp) > 0 else np.nan

            # 如果n-gram surprisal是NaN，使用sentence surprisal
            if np.isnan(srps[0, i]) and not np.isnan(srps[1, i]):
                srps[0, i] = srps[1, i]

        except Exception as e:
            if verbose:
                print(f"Error processing sentence {i}: '{s}', error: {e}")
            srps[0, i] = np.nan
            srps[1, i] = np.nan

    return srps


In [ ]:
print(srp_text(demo_text))


### Interestingness

#### Semantics

In [ ]:
from string import punctuation
from spacy.lang.en import stop_words

punctuations = list(punctuation)
stop_words = stop_words.STOP_WORDS


# Semantic metrics: speed, circuitousness, volume
def text_vec_full(text: str):
    """
    Return `token_vecs` (in full 300 dimensions)
    - Input: `str` (text with `n_lex` lexical words)
    - Output: `np.ndarray` (word vectors `(n_lex,300)`)
    """
    tokens = nlp(text)
    token_vecs = np.array([])
    for token in tokens:
        if token.text not in punctuations:
            if token.text not in stop_words:
                token_vecs = np.append(token_vecs, token.vector)

    token_vecs = token_vecs.reshape(-1, 300)
    return token_vecs


from sklearn.decomposition import PCA


def optimal_n_dim(text: str, max_dim=200, cutoff=0.90):
    """
    Return the optimal `n_dim` of dimension to deduct the output of `text_vec_full()`
    """
    pts = text_vec_full(text)
    pca = PCA(n_components=max_dim)
    word_vectors_pca = pca.fit_transform(pts)
    explained_variance_ratio = pca.explained_variance_ratio_
    cumulative_variance_ratio = np.cumsum(explained_variance_ratio)

    opt_n_dim = np.where(cumulative_variance_ratio >= cutoff)[0][0] + 1
    return opt_n_dim


def text_vec_sub(text: str, n_dim=100, verbose=False):
    """
    Return `token_vecs` (in `n_dim` dimensions)
    - Input: `str` (text with `n_lex` lexical words)
    - Output: `np.ndarray` (word vectors `(n_lex,n_dim)`)
    """
    pts = text_vec_full(text)
    pca = PCA(n_components=n_dim)
    word_vectors_pca = pca.fit_transform(pts)
    explained_variance_ratio = pca.explained_variance_ratio_
    cumulative_variance_ratio = np.cumsum(explained_variance_ratio)

    if verbose:
        print(f"Cumulative variance ratio: {cumulative_variance_ratio[-1]:.4f}")
        return word_vectors_pca
    else:
        return word_vectors_pca


def text_vec_dist(text: str, use_sub=True, n_dim=100, verbose=False):
    """
    Return `all_dist`, `token_dist`, `token_vecs` when `verbose = True`
    - Input: `str` (text with `n_lex` lexical words)
    - Output:
    - `all_dist`: `float`
    - `token_dist`: `np.ndarray (n_lex,n_lex)`
    - `token_vecs`: `np.ndarrat (n_lex,n_dim)`
    """
    if use_sub:
        token_vecs = text_vec_sub(text, n_dim, verbose=False)
    else:
        token_vecs = text_vec_full(text)

    token_dist = distance.cdist(token_vecs, token_vecs)
    all_dist = sum(token_dist[i][i + 1] for i in range(len(token_dist) - 1))

    if verbose:
        return (all_dist, token_dist, token_vecs)
    else:
        return all_dist


from scipy.spatial import distance
import math

def nearest_neighbor_tsp(dist_matrix, return_path=False):
    """
    Use the Nearest Neighbor method to get the Shortest Path (from a Traveling Salesman Problem)
    - Input: `dist_matrix`
    - Output:
    - `total_dist`: `float` (the shortest distance)
    - `path`: `list` (indices of rearranged nodes)
    """

    # `dist_matrix` is a square matrix
    # where `dist_matrix`[i][j] is the distance between city i and city j
    n_rows, n_cols = dist_matrix.shape
    start_city = 0
    # end_city = n_rows - 1
    unvisited = list(range(1, n_rows))

    current_city = start_city
    path = [current_city]

    while unvisited:
        nearest_city = min(
            unvisited,
            key=lambda x: dist_matrix[current_city][x] if x < n_cols else np.inf,
        )
        path.append(nearest_city)
        unvisited.remove(nearest_city)
        current_city = nearest_city

    # path.append(end_city)
    total_distance = sum(
        dist_matrix[path[i]][path[i + 1]] for i in range(len(path) - 1)
    )

    if return_path:
        return (total_distance, path)
    else:
        return total_distance


import numpy as np
import numpy.linalg as la
from scipy.stats.mstats import gmean

def get_min_vol_ellipse(P, tolerance=0.01):
    """ Find the minimum volume ellipsoid which holds all the points

    Based on work by Nima Moshtagh
    http://www.mathworks.com/matlabcentral/fileexchange/9542
    and also by looking at:
    http://cctbx.sourceforge.net/current/python/scitbx.math.minimum_covering_ellipsoid.html
    Which is based on the first reference anyway!

    Here, P is a numpy array of N dimensional points like this:
    P = [[x,y,z,...], <-- one point per line
            [x,y,z,...],
            [x,y,z,...]]

    Returns:
    (center, radii, rotation)

    """
    (N, d) = np.shape(P)
    d = float(d)

    # Q will be our working array
    Q = np.vstack([np.copy(P.T), np.ones(N)])
    QT = Q.T

    # initializations
    err = 1.0 + tolerance
    u = (1.0 / N) * np.ones(N)

    # Khachiyan Algorithm
    while err > tolerance:
        V = np.dot(Q, np.dot(np.diag(u), QT))
        M = np.diag(np.dot(QT , np.dot(np.linalg.inv(V), Q)))    # M the diagonal vector of an NxN matrix
        j = np.argmax(M)
        maximum = M[j]
        step_size = (maximum - d - 1.0) / ((d + 1.0) * (maximum - 1.0))
        new_u = (1.0 - step_size) * u
        new_u[j] += step_size
        err = np.linalg.norm(new_u - u)
        u = new_u

    # center of the ellipse
    center = np.dot(P.T, u)

    # the A matrix for the ellipse
    A = np.linalg.inv(
                    np.dot(P.T, np.dot(np.diag(u), P)) -
                    np.array([[a * b for b in center] for a in center])
                    ) / d
    return A, center

def spd_text(text: str, use_sub=False, verbose=False):
    """
    Get the speed of semantic progression from a text
    """
    _, token_dist, _ = text_vec_dist(text, use_sub=use_sub, verbose=True)
    speed = np.mean(
        np.array([token_dist[i][i + 1] for i in range(len(token_dist) - 1)])
    )

    return speed


def scc_text(text: str, use_sub=False, verbose=False):
    """
    Get the circuitousness of semantic progression from a text
    """
    all_dist, token_dist, _ = text_vec_dist(text, use_sub=use_sub, verbose=True)
    min_dist = nearest_neighbor_tsp(token_dist)
    circuit = all_dist / min_dist

    return circuit


def svl_text(text: str, use_sub=False, verbose=False, tolerance = 0.01) -> float:
    """Gets the volume of the chunk embeddings.

    Args:
        chunk_emb (list): list of chunk embeddings
        tolerance (float, optional): tolerance for the Khachiyan algorithm. Defaults to 0.01.

    Returns:
        float: returns the volume of the chunk embeddings

    See:
        https://github.com/smoorjani/word_embedding_measures/blob/main/utils/algs.py
        https://github.com/smoorjani/word_embedding_measures/blob/main/utils/features.py
    """
    _,_,chunk_emb = text_vec_dist(text, use_sub=use_sub, verbose=True)
    emb_dim = chunk_emb.shape[1]
    P = chunk_emb

    rank = np.linalg.matrix_rank(P, tolerance)
    if rank < emb_dim or (rank == emb_dim and P.shape[0] <= emb_dim):
        tempA = P[1:,:].transpose() - P[0,:].transpose().reshape(-1, 1) @ np.ones((1,P.shape[0] - 1))
        U, S, _ = np.linalg.svd(tempA)
        S1 = U[:,:rank-1]
        tempP = np.vstack([(S1.transpose() @ tempA).transpose(), np.zeros((1, rank-1))])
        A, _ = get_min_vol_ellipse(tempP)
    else:
        A, _ = get_min_vol_ellipse(P)

    U, S, _ = np.linalg.svd(A)
    return 1/gmean(np.sqrt(S))

print(spd_text(demo_text))
print(scc_text(demo_text))
print(svl_text(demo_text))


#### Emotion

In [ ]:
# Emotional metrics: speed, circuitousness, volume
import nltk
nltk.download('vader_lexicon')

from nltk.sentiment.vader import SentimentIntensityAnalyzer

sid = SentimentIntensityAnalyzer()


def text_scores(text: str, roll_mean=True):
    """
    Return the compound score from VADER
    """
    doc = nlp(text)
    scores = []
    for sent in doc.sents:
        scores.append(sid.polarity_scores(sent.text)["compound"])
    scores = np.squeeze(np.array(scores))

    if roll_mean:

        def rolling_mean(x, w):
            return np.convolve(x, np.ones(w) / w, mode="valid")

        try:
            rm_scores = rolling_mean(scores, w=int(len(scores) * 0.05))
        except:
            rm_scores = rolling_mean(scores, w=3)
        return (scores, rm_scores)
    else:
        return scores


def one_dist(arr):
    d_mtx = np.ndarray((len(arr), len(arr)))
    for i, v_i in enumerate(arr):
        for j, v_j in enumerate(arr):
            d_mtx[i, j] = np.abs(v_i - v_j)
    return d_mtx


def epd_text(text: str, use_sub=False, verbose=False):
    """
    Return the speed of emotional progression from a text
    (Use un-smoothed scores by default)
    """
    s, rs = text_scores(text)
    if use_sub:
        d_mtx = one_dist(rs)
    else:
        d_mtx = one_dist(s)

    speed = np.mean(np.array([d_mtx[i][i + 1] for i in range(len(d_mtx) - 1)]))

    return speed


def ecc_text(text: str, use_sub=False, verbose=False):
    """
    Return the circuitousness of emotional progression from a text
    (Use un-smoothed scores by default)
    """
    s, rs = text_scores(text)
    if use_sub:
        d_mtx = one_dist(rs)
    else:
        d_mtx = one_dist(s)

    all_dist = np.sum(np.array([d_mtx[i][i + 1] for i in range(len(d_mtx) - 1)]))
    min_dist = nearest_neighbor_tsp(d_mtx)
    circuit = all_dist / min_dist

    return circuit


def evl_text(text: str, use_sub=False, verbose=False):
    """
    Return the volume of emotional progression from a text
    (Use un-smoothed scores by default)
    """
    s, rs = text_scores(text)
    if use_sub:
        volume = np.sum(np.array(([abs(i) for i in rs])))
    else:
        volume = np.sum(np.array(([abs(i) for i in s])))

    return volume

print(epd_text(demo_text))
print(ecc_text(demo_text))
print(evl_text(demo_text))


### Feature dict

In [ ]:
def get_feature_dict(text: str, collapse=True):
    metrics = dict()

    # Readability
    metrics["AFKG"], metrics["SMOG"], metrics["ARI"] = read_scores(text)
    metrics["LFP"] = lfp_text(text)
    metrics["MDD"] = mdd_text(text)
    metrics["NSRP"], metrics["SSRP"] = srp_text(text)

    if collapse:
        metrics["MDD_mean"] = metrics["MDD"].mean()
        metrics["MDD_std"] = metrics["MDD"].std()
        metrics["NSRP_mean"] = metrics["NSRP"].mean()
        metrics["NSRP_std"] = metrics["NSRP"].std()
        metrics["SSRP_mean"] = metrics["SSRP"].mean()
        metrics["SSRP_std"] = metrics["SSRP"].std()

    # Interestingness
    metrics["SPD"] = spd_text(text)
    metrics["SCC"] = scc_text(text)

    # This two may raise error
    try:
        metrics["SVL"] = svl_text(text)
    except:
        metrics["SVL"] = np.nan

    metrics["EPD"] = epd_text(text)
    metrics["ECC"] = ecc_text(text)
    metrics["EVL"] = evl_text(text)

    return metrics


In [ ]:
idx = 0

print(full_data.sort_values('word_count')['word_count'].values[idx])
demo_text = full_data.sort_values('word_count')['story'].values[idx]

get_feature_dict(demo_text)


## Processing features

In [ ]:
import pickle
import os
import pandas as pd
from tqdm import tqdm
from collections import defaultdict

input_df = full_data
os.makedirs('../outputs', exist_ok=True)
output_file = f"../outputs/{input_df.shape[0]}-all-features-{pythia_size}.pkl"

# Load existing results
if os.path.exists(output_file):
    try:
        with open(output_file, 'rb') as f:
            results = pickle.load(f)
        print(f"Loaded existing results, containing features for {len(results)} texts")
    except Exception as e:
        print(f"Failed to load existing results, starting fresh: {e}")
        results = {}
else:
    results = {}
    print("No existing results file found, starting fresh")

def save_results():
    """Save results to pkl file"""
    try:
        # Save to temporary file first, then rename to avoid file corruption during write
        temp_file = output_file + '.tmp'
        with open(temp_file, 'wb') as f:
            pickle.dump(results, f)

        # Atomic operation: rename temporary file to final file
        os.replace(temp_file, output_file)
        # print(f"Results saved to {output_file} (total {len(results)} texts)")
    except Exception as e:
        print(f"Failed to save results: {e}")

# Calculate progress
total_count = len(input_df)
processed_count = len(results)
remaining_count = total_count - processed_count

print(f"Total texts: {total_count}, Processed: {processed_count}, Remaining: {remaining_count}")

# Use tqdm to show progress
for idx, row in tqdm(input_df.iterrows(), total=total_count, initial=processed_count):
    text_id = row['story_id']

    # Skip already processed texts
    if text_id in results:
        continue

    text = row['story']

    # Check if text is empty
    if pd.isna(text) or str(text).strip() == "":
        results[text_id] = {"error": "empty_text"}
        # Save after processing each text
        save_results()
        continue

    try:
        # Get features
        features = get_feature_dict(text, collapse=True)
        results[text_id] = features
        # Save after processing each text
        save_results()

    except Exception as e:
        results[text_id] = {"error": str(e)}
        save_results()
        print(f"Row {idx} (text_id: {text_id}): {e}")

print(f"\nProcessing completed! Processed {len(results)} texts in total, results saved to {output_file}")

# Statistics
success_count = sum(1 for v in results.values() if 'error' not in v)
error_count = len(results) - success_count
print(f"Success: {success_count}, Errors: {error_count}")
